In [1]:
# Load env variables and create client
import base64
from email.quoprimime import decode

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [7]:
import base64

with open("images/prop7.png", "rb") as f:
    image_bytes = base64.b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    {
        "type": "image",   # ✅ FIXED
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

response = chat(messages)

print(text_from_message(response))

# Property Fire Risk Analysis

**1. Residence identification:**
The primary residence is a light gray-roofed structure located in the center-lower portion of the image, featuring an irregular footprint with multiple roof sections, clearly distinguishable as the largest and only significant building on the heavily wooded property.

**2. Tree overhang analysis:**
Multiple large tree canopies directly overhang substantial portions of the residence roof, with approximately 50-75% of the roof area covered by overhanging branches from mature trees surrounding the structure on all sides, with particularly dense coverage on the northern and eastern sections.

**3. Fire risk assessment:**
The overhanging trees create significant wildfire vulnerability by providing direct pathways for embers to accumulate on the roof, establish multiple fuel bridges between the surrounding dense forest canopy and the structure, and create continuous vegetation contact points that could facilitate fire spread dir

In [9]:
import base64

with open("earth.pdf", "rb") as f:
    file_bytes = base64.b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        }
    },
    {
        "type": "text",
        "text": "How were earth's atmosphere and ocean's formed?"
    },
  ],
)

chat(messages)

Message(id='msg_016GGmNbvDLFP4iZR9deb9c5', container=None, content=[TextBlock(citations=None, text="According to the document, Earth's atmosphere and oceans were formed by **volcanic activity and outgassing**.\n\nThe process worked as follows:\n\n1. **Water vapor** from volcanic sources condensed into the oceans\n\n2. This was **augmented by water and ice** from asteroids, protoplanets, and comets\n\nThe document notes that sufficient water to fill the oceans may have been present on Earth since it formed. Additionally, atmospheric greenhouse gases played a crucial role in keeping the oceans from freezing when the newly forming Sun had only 70% of its current luminosity.\n\nBy 3.5 billion years ago (Ga), Earth's magnetic field was established, which helped prevent the atmosphere from being stripped away by solar wind.", type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_cr